# 04 — Bias & Data Leakage

Pergunta: **o label pode ser previsto por fonte, dominio, autor, periodo, categoria, tamanho, titulo ou origem?**

Severidade: alta quando um atalho afeta grande parte da base ou compartilha noticias entre folds; media quando localizado/condicional; baixa quando a cobertura e pequena.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import load_config, ensure_output_directories
from src.data.make_dataset import build_interim_datasets, load_interim_datasets

cfg = load_config()
ensure_output_directories(cfg)
if not list(Path(cfg["data"]["interim"]).glob("*.parquet")):
    build_interim_datasets()
datasets = {str(frame["dataset"].iloc[0]): frame for frame in load_interim_datasets().values()}
list(datasets)


In [ ]:
from src.analysis.duplicates import cross_dataset_duplicates
from src.analysis.leakage import metadata_label_association, leakage_risk_table, weighted_purity

parts = [metadata_label_association(frame, field, name, min_count=5) for name, frame in datasets.items() for field in ("domain", "author", "category")]
associations = pd.concat([part for part in parts if not part.empty], ignore_index=True)
associations.sort_values(["purity", "total"], ascending=False).head(50)

## O label pode ser descoberto apenas por fonte, autor ou categoria?

In [ ]:
associations.groupby(["dataset", "field"]).apply(lambda table: pd.Series({"weighted_purity": weighted_purity(table), "records_in_pure_groups": table.loc[table["purity"].eq(1), "total"].sum(), "groups": len(table)}), include_groups=False).reset_index()

## As classes cobrem periodos diferentes?

In [ ]:
periods = []
for name, frame in datasets.items():
    dated = frame.dropna(subset=["date_parsed", "label_semantic"])
    if not dated.empty:
        table = dated.groupby("label_semantic")["date_parsed"].agg(["count", "min", "max"]).reset_index()
        table.insert(0, "dataset", name)
        periods.append(table)
pd.concat(periods, ignore_index=True)

## O dataset de origem determina a classe?

In [ ]:
combined = pd.concat(datasets.values(), ignore_index=True)
pd.crosstab(combined["dataset"], combined["label_semantic"], normalize="index")

## Textos semelhantes e tabela de riscos

In [ ]:
cross = cross_dataset_duplicates(datasets)
display(cross.groupby(["match_type", "label_conflict"]).size().rename("groups").reset_index())
leakage_risk_table(datasets, cross).rename(columns={"risk": "Risco", "evidence": "Evidencia", "severity": "Severidade", "recommendation": "Recomendacao"})

## Regras antes de modelar

- Criar grupos por URL, hash e cluster de quase duplicatas.
- Comparar modelos com e sem marcadores/metadados.
- Nao usar split aleatorio por linha.
- Avaliar group split, split temporal e holdout externo descontaminado.